<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

In [1]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu" #Mover tudo para a GPU #Fulcral para treino
print (device)

cuda


### Dados do Modelo

In [2]:
Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8

###


---

### Tokenization


In [21]:
Tokenizer = Tokenizer.from_file ("GAMA")

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset)
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long)
print (tensor [:50])

teste = Tokenizer.encode ("Os Lusiadas são fixes")
print (teste.tokens)
print (teste.ids)

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

74830
74830
tensor([  26, 4359,   23,   32,   30,   69,   13,   16,   13, 4359,    0,   23,
        1526,   94,  679,  119,  527,    0, 2146,   21,    2,    0,  342,  670,
        1060, 2561,  527, 2459,  396,  142,  137, 2852,  114,  206, 4180,  781,
        3292, 2513,  573, 4572,  692,  396, 1997,  660,   38,  541,  113,  362,
         137,   31])
['Os ', 'Lus', 'i', 'adas ', 'são ', 'fi', 'x', 'es']
[280, 427, 46, 788, 610, 174, 59, 103]


###


---

### Batch e Sequence Length

In [4]:
torch.manual_seed (2000)

def Batch (split):

  dados = dados_treino if split == 'treino' else dados_teste

  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  x, y = x.to(device), y.to(device) #Passa dados para a GPU

  return x, y

x, y = Batch('train')

print (x.shape) #(B, T)
print (x[1])

print (y.shape) #(B, T)
print (y[1])

torch.Size([16, 32])
tensor([1061,   89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,
         144,  395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,
         330, 1334,    0, 2560, 1186,  604,  343,  728], device='cuda:0')
torch.Size([16, 32])
tensor([  89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,  144,
         395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,  330,
        1334,    0, 2560, 1186,  604,  343,  728,  763], device='cuda:0')


###


---



### Definição de uma Head

In [5]:
class Head (nn.Module): #Head = Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False)

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf"))

    attention_score = F.softmax (attention_score, dim = -1)

    output = attention_score @ V
    return output


###


---


### Definição de Multi Head

In [6]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


###


---


### Definição de um Block (Pre-LayerNorm Transformer)

Em modelos GPT, um Block é:

- Layer Norm
- Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network
- Residual Connection (Add)

In [7]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension)
    self.Masked_Multi_Head_Attention = Multi_Head()


    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension)
    self.FeedForward_NeuralNet = nn.Sequential (
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )


  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x


###


---


### Modelo Adamastor-GPT

In [8]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension)
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension)

    self.Blocks = nn.Sequential (
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Return de um tensor (4, 8, 5000)



###


---


### Treino do Modelo (Super-Importante)


###### Calcular a Loss

In [13]:
#https://github.com/karpathy/nanoGPT/blob/master/train.py
@torch.no_grad()
def loss_calc (modelo, loop_eval = 500):

    output = {}
    modelo.eval() #Desliga modo de treino

    for split in ['train', 'val']:

        losses = torch.zeros(loop_eval)

        for k in range(loop_eval):

            x, y = Batch(split)
            logits = modelo(x)

            B, T, V = logits.shape

            loss = loss_function (
                logits.view(B*T, V),
                y.view(B*T)
            )

            losses[k] = loss.item()

        output[split] = losses.mean()

    modelo.train()
    return output

In [14]:
def training (modelo, steps):

  modelo.train() #Ativa o treino

  for i in range (steps):

    x, y = Batch('train')

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss = loss_function (
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 1000 == 0: #Printa de 1000 em 1000

      losses = loss_calc (modelo)
      print(f"Step {i} | Train Loss = {losses['train']:.4f} | Validation Loss : {losses['val']:.4f}")


###


---

### Geração de Texto

In [16]:
def gerar_texto (modelo, max_tokens):

#É possível adicionar temperatura ao modelo, que no fundo é a creatividade dele

    device = next(modelo.parameters()).device  #Vai buscar o modelo à GPU
    tokens = torch.zeros((1, 1), dtype = torch.long, device = device)
    tokens = tokens.to(device)

    modelo.eval() #Desliga o treino

    with torch.no_grad(): #Desliga o gradiente
      for _ in range(max_tokens):

        contexto_max = tokens[:, -Sequence_Length:] #Alimento o modelo com o máximo de tokens possíveis, não posso dar um contexto maior do que daquele com que treinou

        logits = modelo(contexto_max) #(B,T,VOCAB_SIZE)
        logits = logits[:, -1, :] #O que vem depois do último token que gerou ?

        probabilidade = F.softmax(logits, dim = -1) #Aplica normalização

        next_token = torch.multinomial(probabilidade, num_samples = 1) #Seleciona um token aleatoriamente ponderado

        tokens = torch.cat([tokens, next_token], dim = 1) #Adiciona a uma lista com todos os outros tokens

      text = Tokenizer.decode(tokens[0].tolist()) #Decode do token para palavra ou sub palavra

      print(text)


###


---

### Adamastor-GPT Treino e Inferência

In [15]:
modelo = ADAMASTOR_GPT().to(device)
optimizer = torch.optim.AdamW (modelo.parameters(), lr = 1e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.

training (modelo, 25000)



Step 0 | Train Loss = 8.8879 | Validation Loss : 8.8877
Step 1000 | Train Loss = 7.7862 | Validation Loss : 7.7880
Step 2000 | Train Loss = 7.7002 | Validation Loss : 7.6911
Step 3000 | Train Loss = 7.5889 | Validation Loss : 7.5901
Step 4000 | Train Loss = 7.4127 | Validation Loss : 7.4080
Step 5000 | Train Loss = 7.1764 | Validation Loss : 7.1731
Step 6000 | Train Loss = 6.9026 | Validation Loss : 6.9047
Step 7000 | Train Loss = 6.6198 | Validation Loss : 6.6262
Step 8000 | Train Loss = 6.3694 | Validation Loss : 6.3844
Step 9000 | Train Loss = 6.1673 | Validation Loss : 6.1588
Step 10000 | Train Loss = 5.9659 | Validation Loss : 5.9727
Step 11000 | Train Loss = 5.8004 | Validation Loss : 5.7926
Step 12000 | Train Loss = 5.6421 | Validation Loss : 5.6483
Step 13000 | Train Loss = 5.5013 | Validation Loss : 5.5039
Step 14000 | Train Loss = 5.3687 | Validation Loss : 5.3683
Step 15000 | Train Loss = 5.2526 | Validation Loss : 5.2421
Step 16000 | Train Loss = 5.1288 | Validation Loss : 

In [24]:
gerar_texto (modelo, 200)


  L ando;  Em  derred o ou  ro vol ta  com que  queim -te  ilustre,  duro e  os  nem  de  D da  morte  no  teu  mur o a  divi da  últim a  poder ás,  Ci pr at eia  bem  ao  porto  cos  instrum es ! « Esta  bo esm os  famoso  G ó ,  de  Lisbo o.  G idade  d' alm ente.  Não  tens  que,  os filhos  terão  gu a. « Oh ,  no  descend entes  naveg ar  de Cristo  faz  ac e,  se mov eu.  As  ped a o  corr ente  ( costume  eiros  Do  velh desse  dos  do  muito  que  na terra  A quem  segundo  h ante  em quanto  em  cores  Portugues o se  mostr ando,  doutr o, Se  vão  pôde  pos da e  em terra  Vel os ,
 
  figura  cr er  ao  corpo  humano  embar corr o  na m es .»  Isto  desej o, que  direi ta a  justa  pret end eu  pela  areia  guerra,  Agora,  a gente  n o, e  Que  cri es.  Vem  ao  dur o,  maravilh a, Que  acab o,  a quem  que  tu  se est os s adas,  É  est as e outras  de  des pe de T í monte  ao  alto  Rei  que a  Marte  não  cr édi a p anha  Que  não  dão  as  ram o  princ es ecem  a s 

In [25]:
print (modelo.state_dict())
for param in modelo.state_dict():
  print (param)

OrderedDict({'Embedding.weight': tensor([[ 1.1582, -1.2619, -0.0234,  ...,  0.5197,  0.5327, -0.5349],
        [ 1.3830, -0.0070, -0.8032,  ..., -2.3529,  0.2384, -0.8037],
        [-0.0375, -1.3306,  1.5230,  ...,  1.5828, -0.6616,  1.2007],
        ...,
        [-0.2579,  0.5275, -1.1094,  ..., -1.7861,  1.0205,  0.6765],
        [ 0.0635, -1.1006, -0.3236,  ..., -0.7051, -0.6785, -0.3618],
        [-0.7748,  1.5131, -0.4929,  ...,  0.3141, -0.9974, -2.0211]],
       device='cuda:0'), 'Positional_Encoding.weight': tensor([[-0.0816, -0.2457, -0.6498,  ...,  0.3701, -1.5666,  1.0560],
        [-0.9630, -0.4908, -0.3483,  ..., -0.4211,  1.4156, -0.1088],
        [ 0.4054, -1.6450,  0.4111,  ...,  0.3327,  1.4956,  0.4458],
        ...,
        [ 0.5212,  1.8230, -0.1454,  ...,  0.3852,  0.6001,  0.4759],
        [-0.2205,  1.1368,  0.6370,  ...,  0.0985, -1.2492,  1.4548],
        [ 0.7101, -0.9851,  0.3577,  ...,  0.3887,  0.2443, -0.2741]],
       device='cuda:0'), 'Blocks.0.LayerNorm

In [26]:
print (sum(p.numel() for p in modelo.parameters()))
#save = torch.save(modelo.state_dict(), "Adamastor-GPT.pth")

846216


O teu ADAMASTOR_GPT é:

um Transformer decoder-only (tipo GPT)
com:
4 camadas (blocos)
8 cabeças de atenção
embeddings de tamanho 64
vocabulário de 5000 tokens
contexto máximo de 32 tokens

FAZER ESTA LEITURA

interpretar loss value, o que o valor quer dizer ? <br>

CPU do PC vs TPU - tempo a treinar o modelo e resultados obtidos

Dropout é uma técnica de regularização usada para evitar overfitting.

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster